# Certified MNIST addition consistency

Predict two digit classes and their sum using three cross-entropy heads. The general property requires the predicted sum to equal the sum of the predicted operands throughout the input box. **Consistency does not imply recognition correctness.** Raw pixels are clipped before normalization.

This notebook executes two paired baselines on a **synthetic two-epoch CPU fixture**. It downloads no MNIST data and produces no benchmark evidence. See [setup](index.md) before starting.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "case_studies").is_dir() and (p / "pyproject.toml").is_file())
sys.path.insert(0, str(REPO))
WORK = Path(tempfile.mkdtemp(prefix="ctrain-case-study-"))
print("Fresh outputs:", WORK)


## Run paired baselines

The CE-only and property-training arms share initialization and seeded batches. Training uses `PropertyModelWrapper.train_model`.

In [ ]:
output = WORK / "mnist_addition"
command = [sys.executable, "-m", "case_studies.mnist_addition.run",
           "--smoke", "--device", "cpu", "--output", str(output)]
completed = subprocess.run(command, cwd=REPO, check=True, capture_output=True, text=True)
result = json.loads((output / "results.json").read_text())
print(json.dumps(result["results"], indent=2))

## Keep accuracy and consistency separate

The certification metric concerns arithmetic consistency within the declared perturbation boxes. Both operands and the sum can be consistently wrong.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
arms = list(result["results"])
metrics = ["Triple accuracy", "Sampled consistency", "Certified consistency"]
fig, ax = plt.subplots(figsize=(8, 4))
positions = np.arange(len(metrics))
for i, arm in enumerate(arms):
    row = result["results"][arm]
    values = [row["test_metrics"]["triple_accuracy"],
              row["test_metrics"]["empirical_consistent_fraction"], row["test_certified_fraction"]]
    ax.bar(positions + (i-.5)*.35, values, width=.35, label=arm)
ax.set(xticks=positions, xticklabels=metrics, ylim=(0, 1), ylabel="Fraction", title="Synthetic execution check")
ax.legend()
plt.show()

## Run real MNIST and HPO

```bash
python -m case_studies.mnist_addition.run --config case_studies/mnist_addition/config.json --download --device cuda --output runs/case_studies/addition_real
python -m case_studies.mnist_addition.hpo --method SABR --trials 20 --device cuda --output runs/case_studies/addition_hpo/seed_0
```

Use `--device cpu` if needed. Download is explicit in the baseline command. HPO uses a deterministic training holdout before pairing and evaluates the official test split only after selection. Repeat seeds 1 and 2 before pooling fronts. SABR tunes its half-width ratio; MTL-IBP tunes alpha. Baselines keep their configured options.

The separate prediction-stability diagnostic examines a selected trial's saved CROWN cover and searches for changed triples. It distinguishes fixed consistent predictions from flexible arithmetic consistency. [Full protocol, comparison exports and diagnostic](https://github.com/ADA-research/CTRAIN/blob/dev/case_studies/mnist_addition/README.md).